In [32]:
import csv
from pathlib import Path

In [33]:
out_dir = Path("../data")
sales_path = out_dir / "sales.csv"
customers_path = out_dir / "customers.csv"

In [34]:
sales_rows = [
    ["Sale_ID", "Customer_ID", "Product", "Quantity", "Sale_Amount"],
    ["S001", "C001", "Laptop Stand", 1, 39.99],
    ["S002", "C002", "Wireless Mouse", 2, 49.98],
    ["S003", "C003", "USB-C Hub", 1, 54.50],
    ["S004", " c004 ", "Mechanical Keyboard", 1, 89.99],   # spaces + lower case
    ["S005", "C001", "Webcam", 1, 64.99],
    ["S006", "C005", "Laptop Sleeve", 2, 45.00],
    ["S007", "C006", "Monitor", 1, 219.00],
    ["S008", "C007 ", "USB-C Cable", 3, 29.97],            # trailing space
    ["S009", "C008", "Desk Lamp", 1, 34.95],
    ["S010", "C009", "Headset", 1, 79.90],
    ["S011", "C002", "Laptop Stand", 1, 39.99],
    ["S012", "C006", "Wireless Mouse", 1, 24.99],
    ["S013", "C011", "Portable SSD", 1, 109.00],           # no matching customer
    ["S014", "C003", "USB-C Cable", 2, 19.98],
    ["S015", "C008", "Webcam", 1, 64.99],
]

customers_rows = [
    ["Customer_ID", "Customer_Name", "Location", "Contact_Information"],
    ["C001", "Anna Keller", "Berlin", "anna.keller@example.com"],
    ["C002", "David Schmidt", "Hamburg", "david.schmidt@example.com"],
    ["C003", "Sara Weber", "Munich", "sara.weber@example.com"],
    ["C004", "Omar Haddad", "Cologne", "omar.haddad@example.com"],
    ["C005", "Lisa Braun", "Frankfurt", "lisa.braun@example.com"],
    ["C006", "Max Fischer", "Berlin", "max.fischer@example.com"],
    ["C006", "Max Fischer", "Berlin", "max.fischer@oldmail.com"],  # duplicate key
    ["C007", "Nina Hoffmann", "Dresden", "nina.hoffmann@example.com"],
    ["C008", "Leon Wagner", "Leipzig", "leon.wagner@example.com"],
    ["C009", "Mia Becker", "Stuttgart", "mia.becker@example.com"],
    ["C010", "Jonas Koch", "Bremen", "jonas.koch@example.com"],    # customer with no sales
]


In [35]:
with sales_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(sales_rows)

with customers_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(customers_rows)

In [36]:
print(f"Created: {sales_path}")
print(f"Created: {customers_path}")
print(f"Sales rows (excluding header): {len(sales_rows)-1}")
print(f"Customer rows (excluding header): {len(customers_rows)-1}") 

Created: ../data/sales.csv
Created: ../data/customers.csv
Sales rows (excluding header): 15
Customer rows (excluding header): 11


In [37]:
import pandas as pd
sales = pd.read_csv('../data/sales.csv')
customers = pd.read_csv('../data/customers.csv')
sales, customers.head(), sales.shape, customers.shape


(   Sale_ID Customer_ID              Product  Quantity  Sale_Amount
 0     S001        C001         Laptop Stand         1        39.99
 1     S002        C002       Wireless Mouse         2        49.98
 2     S003        C003            USB-C Hub         1        54.50
 3     S004       c004   Mechanical Keyboard         1        89.99
 4     S005        C001               Webcam         1        64.99
 5     S006        C005        Laptop Sleeve         2        45.00
 6     S007        C006              Monitor         1       219.00
 7     S008       C007           USB-C Cable         3        29.97
 8     S009        C008            Desk Lamp         1        34.95
 9     S010        C009              Headset         1        79.90
 10    S011        C002         Laptop Stand         1        39.99
 11    S012        C006       Wireless Mouse         1        24.99
 12    S013        C011         Portable SSD         1       109.00
 13    S014        C003          USB-C Cable    

In [38]:
# raw merge counts
counts_raw = {}
for how in ['inner','left','right','outer']:
    m = sales.merge(customers, on='Customer_ID', how=how, indicator=True)
    counts_raw[how] = (len(m), m['_merge'].value_counts().to_dict())

In [39]:
# normalize IDs
sales2=sales.copy()
cust2=customers.copy()
for df in (sales2,cust2):
    df['Customer_ID'] = df['Customer_ID'].str.strip().str.upper()

In [40]:
left_merge = pd.merge(
    sales,
    customers,
    on="Customer_ID",
    how="left"
)

print(left_merge)
print(left_merge.shape)

   Sale_ID Customer_ID              Product  Quantity  Sale_Amount  \
0     S001        C001         Laptop Stand         1        39.99   
1     S002        C002       Wireless Mouse         2        49.98   
2     S003        C003            USB-C Hub         1        54.50   
3     S004       c004   Mechanical Keyboard         1        89.99   
4     S005        C001               Webcam         1        64.99   
5     S006        C005        Laptop Sleeve         2        45.00   
6     S007        C006              Monitor         1       219.00   
7     S007        C006              Monitor         1       219.00   
8     S008       C007           USB-C Cable         3        29.97   
9     S009        C008            Desk Lamp         1        34.95   
10    S010        C009              Headset         1        79.90   
11    S011        C002         Laptop Stand         1        39.99   
12    S012        C006       Wireless Mouse         1        24.99   
13    S012        C0

In [41]:
right_merge = pd.merge(
    sales,
    customers,
    on="Customer_ID",
    how="right"
)

print(right_merge)

   Sale_ID Customer_ID         Product  Quantity  Sale_Amount  Customer_Name  \
0     S001        C001    Laptop Stand       1.0        39.99    Anna Keller   
1     S005        C001          Webcam       1.0        64.99    Anna Keller   
2     S002        C002  Wireless Mouse       2.0        49.98  David Schmidt   
3     S011        C002    Laptop Stand       1.0        39.99  David Schmidt   
4     S003        C003       USB-C Hub       1.0        54.50     Sara Weber   
5     S014        C003     USB-C Cable       2.0        19.98     Sara Weber   
6      NaN        C004             NaN       NaN          NaN    Omar Haddad   
7     S006        C005   Laptop Sleeve       2.0        45.00     Lisa Braun   
8     S007        C006         Monitor       1.0       219.00    Max Fischer   
9     S012        C006  Wireless Mouse       1.0        24.99    Max Fischer   
10    S007        C006         Monitor       1.0       219.00    Max Fischer   
11    S012        C006  Wireless Mouse  

In [42]:
outer_merge = pd.merge(
    sales,
    customers,
    on="Customer_ID",
    how="outer"
)

print(outer_merge)

   Sale_ID Customer_ID              Product  Quantity  Sale_Amount  \
0     S004       c004   Mechanical Keyboard       1.0        89.99   
1     S001        C001         Laptop Stand       1.0        39.99   
2     S005        C001               Webcam       1.0        64.99   
3     S002        C002       Wireless Mouse       2.0        49.98   
4     S011        C002         Laptop Stand       1.0        39.99   
5     S003        C003            USB-C Hub       1.0        54.50   
6     S014        C003          USB-C Cable       2.0        19.98   
7      NaN        C004                  NaN       NaN          NaN   
8     S006        C005        Laptop Sleeve       2.0        45.00   
9     S007        C006              Monitor       1.0       219.00   
10    S007        C006              Monitor       1.0       219.00   
11    S012        C006       Wireless Mouse       1.0        24.99   
12    S012        C006       Wireless Mouse       1.0        24.99   
13     NaN        C0

In [43]:
outer_merge = pd.merge(
    sales,
    customers,
    on="Customer_ID",
    how="outer",
    indicator=True
)

In [44]:
# This immediately shows problematic records.
outer_merge[
    outer_merge["_merge"] != "both"
]

,Sale_ID,Customer_ID,Product,Quantity,Sale_Amount,Customer_Name,Location,Contact_Information,_merge
0,S004,c004,Mechanical Keyboard,1.0,89.99,NaN,NaN,NaN,left_only
7,NaN,C004,NaN,NaN,NaN,Omar Haddad,Cologne,omar.haddad@example.com,right_only
13,NaN,C007,NaN,NaN,NaN,Nina Hoffmann,Dresden,nina.hoffmann@example.com,right_only
14,S008,C007,USB-C Cable,3.0,29.97,NaN,NaN,NaN,left_only
18,NaN,C010,NaN,NaN,NaN,Jonas Koch,Bremen,jonas.koch@example.com,right_only
19,S013,C011,Portable SSD,1.0,109.00,NaN,NaN,NaN,left_only


In [45]:
print(sales["Customer_ID"].unique())
print(customers["Customer_ID"].unique())

<StringArray>
[  'C001',   'C002',   'C003', ' c004 ',   'C005',   'C006',  'C007 ',
   'C008',   'C009',   'C011']
Length: 10, dtype: str
<StringArray>
['C001', 'C002', 'C003', 'C004', 'C005', 'C006', 'C007', 'C008', 'C009',
 'C010']
Length: 10, dtype: str


In [46]:
# Clean both datasets:
sales["Customer_ID"] = (
    sales["Customer_ID"]
    .str.strip()
    .str.upper()
)

customers["Customer_ID"] = (
    customers["Customer_ID"]
    .str.strip()
    .str.upper()
)

In [47]:
# Check duplicates before merging
customers[
    customers.duplicated(
        subset="Customer_ID",
        keep=False
    )
]

,Customer_ID,Customer_Name,Location,Contact_Information
5,C006,Max Fischer,Berlin,max.fischer@example.com
6,C006,Max Fischer,Berlin,max.fischer@oldmail.com


In [48]:
# we can keep the first record:
customers_clean = customers.drop_duplicates(
    subset="Customer_ID",
    keep="first"
)

In [49]:
# Now perform a validated Left Join
integrated = pd.merge(
    sales,
    customers_clean,
    on="Customer_ID",
    how="left",
    indicator=True,
    validate="many_to_one"
)
# Many sales may belong to one customer, 
# but each customer ID should occur only once in the customer table.

In [50]:
print(integrated.shape)

(15, 9)


In [51]:
# Find customers without sales
audit_merge = pd.merge(
    sales,
    customers_clean,
    on="Customer_ID",
    how="outer",
    indicator=True,
    validate="many_to_one"
)

In [52]:
audit_merge[
    audit_merge["_merge"] == "right_only"
]
# C010 exists in the customer database but has no sales transaction.

,Sale_ID,Customer_ID,Product,Quantity,Sale_Amount,Customer_Name,Location,Contact_Information,_merge
14,NaN,C010,NaN,NaN,NaN,Jonas Koch,Bremen,jonas.koch@example.com,right_only
